# VisuModel : entraînement YOLO (une seule classe : imperfection)

Avant de lancer :
1. **Add Input** → le dataset `imperfections` (le zip fait par `yolo_merge_classes.py`, avec `train/`, `valid/`, `test/`). Ne laisse que celui-là en entrée.
2. **Settings → Accelerator → GPU T4**.
3. **Settings → Internet → On** (pour installer Ultralytics et télécharger le modèle de départ).
4. **Run All**. À la fin, télécharge `yolo_model.pt` dans l'onglet **Output**.

In [ ]:
!pip -q install ultralytics
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (activez-le dans Settings)')

In [ ]:
import glob, os, zipfile

# 1. Ce qui est réellement en entrée (pour comprendre si ça ne marche pas)
print('/kaggle/input contient :', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'RIEN')
for d, subs, files in os.walk('/kaggle/input'):
    if d.count('/') <= 5:
        print(d, '| dossiers :', subs[:8], '| fichiers :', files[:3])

# 2. Si Kaggle a laissé une archive .zip telle quelle, on la décompresse
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/working/data')

# 3. Dossier qui contient train, valid et test, chacun avec un sous-dossier images
def ok(d):
    return all(os.path.isdir(f'{d}/{s}/images') for s in ('train', 'valid', 'test'))
data = next((d for root in ('/kaggle/input', '/kaggle/working/data') for d, subs, _ in os.walk(root) if ok(d)), None)
assert data, "Dataset introuvable : regardez la liste ci-dessus. Il faut Add Input avec un dossier contenant train/images, valid/images, test/images."
assert len(os.listdir(f'{data}/train/images')) > 0
print('dataset :', data, {s: len(os.listdir(f'{data}/{s}/images')) for s in ('train', 'valid', 'test')})

open('/kaggle/working/acne.yaml', 'w').write(f'''path: {data}
train: train/images
val: valid/images
test: test/images
names:
  0: imperfection
''')

In [ ]:
from ultralytics import YOLO

MODEL = 'yolo11n.pt'   # yolo11s.pt = plus précis mais ~2,5x plus lent en temps réel sur ton PC
model = YOLO(MODEL)
model.train(
    data='/kaggle/working/acne.yaml', epochs=100, patience=30, imgsz=1024, batch=8, device=0,
    project='/kaggle/working/runs', name='acne', exist_ok=True, workers=2, cache='ram',
    hsv_h=0.01, hsv_s=0.5, hsv_v=0.4,   # peu de variation de teinte : la rougeur est le signal
    degrees=15, scale=0.5, flipud=0.0, mosaic=1.0,
)

In [ ]:
import shutil

best = '/kaggle/working/runs/acne/weights/best.pt'
shutil.copy(best, '/kaggle/working/yolo_model.pt')
m = YOLO(best).val(data='/kaggle/working/acne.yaml', split='test', imgsz=1024)  # mesure finale sur le test, une seule fois
print(f'mAP50 = {m.box.map50:.3f} | mAP50-95 = {m.box.map:.3f}')
for i, name in m.names.items():
    print(f'{name:12s} précision {m.box.p[i]:.2f}  rappel {m.box.r[i]:.2f}  mAP50 {m.box.ap50[i]:.2f}')
print('-> télécharge /kaggle/working/yolo_model.pt dans Output')